<a href="https://colab.research.google.com/github/Sanath-cmd/Internship_ITT/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [144]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [145]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA


In [146]:
data = pd.read_csv('/content/drive/MyDrive/heart.csv')
data.columns
data.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [147]:
encoder = OneHotEncoder(sparse_output=False)
encoded_values = encoder.fit_transform(data[['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']])
new_cols = encoder.get_feature_names_out(['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'])
df_encoded = pd.DataFrame(encoded_values, columns= new_cols)
df_final = pd.concat([data.drop(['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'],axis = 1), df_encoded], axis=1)
df_final.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_F,Sex_M,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,49,160,180,0,156,1.0,1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,37,130,283,0,98,0.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
3,48,138,214,0,108,1.5,1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,54,150,195,0,122,0.0,0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [148]:
X = df_final.drop('HeartDisease', axis=1)
y = df_final['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state= 42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
param_grid1 = {'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
               'C': [0.1, 1, 10, 100, 1000]
               }
grid1 = GridSearchCV(SVC(), param_grid1)
grid1.fit(X_train, y_train)
print(grid1.best_params_)

param_grid2 = {'max_iter':[100, 500, 1000, 10000]}
grid2 = GridSearchCV(LogisticRegression(), param_grid2)
grid2.fit(X_train, y_train)
print(grid2.best_params_)

param_grid3 = {'n_estimators':[100, 500, 1000, 10000]}
grid3 = GridSearchCV(RandomForestClassifier(), param_grid3)
grid3.fit(X_train, y_train)
print(grid3.best_params_)

In [149]:
#SVM
svm_model = SVC(kernel = 'linear', C= 0.1)
svm_model.fit(X_train, y_train)
svm_score = svm_model.score(X_test, y_test)
svm_score

0.8586956521739131

In [150]:
lr_model = LogisticRegression(max_iter=100)
lr_model.fit(X_train, y_train)
lr_score = lr_model.score(X_test, y_test)
lr_score

0.8532608695652174

In [151]:
rf_model = RandomForestClassifier(n_estimators = 100, random_state=42)
rf_model.fit(X_train, y_train)
rf_score = rf_model.score(X_test, y_test)
rf_score

0.8804347826086957

In [152]:
pca = PCA(0.95)
df_final = pca.fit_transform(df_final)
df_final = pd.DataFrame(df_final)
df_final

,0,1
0,92.311918,29.451820
1,-17.143261,13.740174
2,81.907273,-38.211654
3,13.654158,-28.752894
4,-4.347839,-18.083419
...,...,...
913,64.489986,-1.442287
914,-5.473411,-0.834123
915,-69.004825,-17.343492
916,39.207765,33.594563


In [153]:
X_pca = df_final
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)


In [154]:
rf_model = RandomForestClassifier(n_estimators = 100, random_state=42)
rf_model.fit(X_train, y_train)
rf_score = rf_model.score(X_test, y_test)
rf_score

0.6847826086956522

In [155]:
svm_model = SVC(kernel = 'linear', C= 0.1)
svm_model.fit(X_train, y_train)
svm_score = svm_model.score(X_test, y_test)
svm_score


0.6358695652173914

In [156]:
lr_model = LogisticRegression(max_iter=100)
lr_model.fit(X_train, y_train)
lr_score = lr_model.score(X_test, y_test)
lr_score

0.6358695652173914